# One-Hot Encoding Kata dan Karakter

**Buku:** *Deep Learning dengan Python* — Bab 6, Deep Learning untuk Teks dan Sekuens

Cara paling dasar mengubah token menjadi vektor: implementasi manual, utilitas Keras, dan *hashing trick*.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `keras.preprocessing.text.Tokenizer` (dihapus di Keras 3) → lapisan `TextVectorization`.
- **Bug reprodusibilitas asli:** `hash()` Python diacak per proses (`PYTHONHASHSEED`), sehingga indeks berubah setiap kali dijalankan. Diganti `hashlib` dan lapisan `Hashing` Keras.
- Catatan: listing 6.2 versi cetak buku asli memetakan indeks→karakter (terbalik); notebook ini memakai karakter→indeks.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. One-hot tingkat kata (manual)

In [ ]:
import numpy as np

sampel = ["Kucing duduk di atas tikar.", "Anjing makan pekerjaan rumahku."]
indeks_token = {}
for s in sampel:
    for kata in s.split():
        if kata not in indeks_token:
            indeks_token[kata] = len(indeks_token) + 1   # indeks 0 tidak dipakai

PANJANG_MAKS = 10
hasil = np.zeros((len(sampel), PANJANG_MAKS, max(indeks_token.values()) + 1))
for i, s in enumerate(sampel):
    for j, kata in list(enumerate(s.split()))[:PANJANG_MAKS]:
        hasil[i, j, indeks_token[kata]] = 1.0
print(indeks_token); print(hasil.shape)

## 2. One-hot tingkat karakter (manual)

In [ ]:
import string

karakter = string.printable
indeks_karakter = dict(zip(karakter, range(1, len(karakter) + 1)))   # karakter -> indeks
PANJANG_MAKS = 50
hasil = np.zeros((len(sampel), PANJANG_MAKS, max(indeks_karakter.values()) + 1))
for i, s in enumerate(sampel):
    for j, c in enumerate(s[:PANJANG_MAKS]):
        hasil[i, j, indeks_karakter[c]] = 1.0
print(hasil.shape)

## 3. Dengan Keras: `TextVectorization`
Menangani normalisasi huruf kecil, penghapusan tanda baca, dan pembatasan kosakata.

In [ ]:
from keras import layers

vektorisasi_int = layers.TextVectorization(max_tokens=1000, output_mode="int")
vektorisasi_int.adapt(sampel)
print("Kosakata:", vektorisasi_int.get_vocabulary())
print("Sekuens indeks:\n", keras.ops.convert_to_numpy(vektorisasi_int(sampel)))

vektorisasi_mh = layers.TextVectorization(max_tokens=1000, output_mode="multi_hot")
vektorisasi_mh.adapt(sampel)
print("Matriks multi-hot:", keras.ops.convert_to_numpy(vektorisasi_mh(sampel)).shape)

## 4. *Hashing trick*
Tidak perlu menyimpan indeks kata; risikonya **tabrakan hash** bila dimensi terlalu kecil.

In [ ]:
import hashlib

def indeks_hash(kata, dimensi):
    # hashlib deterministik; hash() bawaan Python TIDAK (berubah tiap sesi)
    return int(hashlib.md5(kata.encode("utf-8")).hexdigest(), 16) % dimensi

DIMENSI, PANJANG_MAKS = 1000, 10
hasil = np.zeros((len(sampel), PANJANG_MAKS, DIMENSI))
for i, s in enumerate(sampel):
    for j, kata in list(enumerate(s.split()))[:PANJANG_MAKS]:
        hasil[i, j, indeks_hash(kata, DIMENSI)] = 1.0

hashing = layers.Hashing(num_bins=DIMENSI)                      # versi lapisan Keras
print(keras.ops.convert_to_numpy(hashing(np.array([["kucing"], ["anjing"]]))))